# 순서와 합성

여러 미들웨어의 before·after·wrap 훅은 어떤 순서로 합성될까요?

## 실행 환경 준비

실습 파일은 `labs/part6/ch22-middleware/notebooks/07-순서와-합성.ipynb`입니다. Python 3.12 새 커널에서 위에서부터 실행합니다.

## 패키지 버전 확인

In [1]:
# 이 셀은 패키지 버전 확인 작업을 수행하고 확인할 값을 출력합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langgraph==1.2.11" "python-dotenv==1.2.3"
from importlib.metadata import version
print("langchain", version("langchain"))
print("langchain-openai", version("langchain-openai"))
print("langgraph", version("langgraph"))

Note: you may need to restart the kernel to use updated packages.
langchain 1.4.0
langchain-openai 1.6.0
langgraph 1.2.11


## 결정적 모델 준비

In [2]:
# 이 셀은 결정적 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
# 정해 둔 응답으로 미들웨어 실행 순서를 재현합니다.
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
class ScriptedModel(BaseChatModel):
    replies: list[AIMessage]
    index: int = 0
    @property
    def _llm_type(self) -> str: return "scripted-model"
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        reply = self.replies[min(self.index, len(self.replies) - 1)]
        self.index += 1
        return ChatResult(generations=[ChatGeneration(message=reply)])
    def bind_tools(self, tools: Any, *, tool_choice: Any = None, **kwargs: Any) -> "ScriptedModel":
        return self
@tool
def lookup_hours(day: str) -> str:
    """자료실 운영 시간을 요일로 조회합니다."""
    return "평일 09:00~20:00" if day == "평일" else "토요일 10:00~17:00"

## 태그 미들웨어 정의

In [3]:
# 이 셀은 태그 미들웨어 정의 작업을 수행하고 확인할 값을 출력합니다.
ORDER=[]
class Tagged(AgentMiddleware):
    def __init__(self, tag: str) -> None: self.tag=tag
    @property
    def name(self) -> str: return f"Tagged-{self.tag}"
    def before_agent(self, state: Any, runtime: Any) -> None: ORDER.append(f"before_agent:{self.tag}")
    def before_model(self, state: Any, runtime: Any) -> None: ORDER.append(f"before_model:{self.tag}")
    def wrap_model_call(self, request: Any, handler: Any) -> Any:
        ORDER.append(f"wrap_in:{self.tag}"); result=handler(request); ORDER.append(f"wrap_out:{self.tag}"); return result
    def after_model(self, state: Any, runtime: Any) -> None: ORDER.append(f"after_model:{self.tag}")
    def after_agent(self, state: Any, runtime: Any) -> None: ORDER.append(f"after_agent:{self.tag}")

## A B C 순서로 실행

In [4]:
# 이 셀은 A B C 순서로 실행 작업을 수행하고 확인할 값을 출력합니다.
ordered=create_agent(ScriptedModel(replies=[AIMessage("완료")]),tools=[],middleware=[Tagged("A"),Tagged("B"),Tagged("C")])
ordered.invoke({"messages":[{"role":"user","content":"순서 확인"}]})
for item in ORDER: print(item)

before_agent:A
before_agent:B
before_agent:C
before_model:A
before_model:B
before_model:C
wrap_in:A
wrap_in:B
wrap_in:C
wrap_out:C
wrap_out:B
wrap_out:A
after_model:C
after_model:B
after_model:A
after_agent:C
after_agent:B
after_agent:A


## 구간별 순서 검사

In [5]:
# 이 셀은 구간별 순서 검사 작업을 수행하고 확인할 값을 출력합니다.
expected=[
"before_agent:A","before_agent:B","before_agent:C","before_model:A","before_model:B","before_model:C",
"wrap_in:A","wrap_in:B","wrap_in:C","wrap_out:C","wrap_out:B","wrap_out:A",
"after_model:C","after_model:B","after_model:A","after_agent:C","after_agent:B","after_agent:A"]
assert ORDER==expected
print("order_match:",ORDER==expected)

order_match: True


## 합성 표 출력

In [6]:
# 이 셀은 합성 표 출력 작업을 수행하고 확인할 값을 출력합니다.
print({"before":"A→B→C","wrap":"A(B(C(model)))","after":"C→B→A","estimated_cost_usd":0.0})

{'before': 'A→B→C', 'wrap': 'A(B(C(model)))', 'after': 'C→B→A', 'estimated_cost_usd': 0.0}
